In [1]:
from pathlib import Path

import joblib
import pandas as pd

from sklearn.model_selection import train_test_split

project_root = Path(
    r"D:\PROGRAMMING\NetGuard AI\netguard_ai"
)

train_df = pd.read_csv(
    project_root / "data/raw/UNSW_NB15_training-set.csv"
)

binary_artifact = joblib.load(
    project_root / "models/binary_rf_baseline.joblib"
)

feature_columns = binary_artifact["input_columns"]

# Keep unique normal feature patterns
normal_data = (
    train_df.loc[train_df["label"] == 0, feature_columns]
    .drop_duplicates()
    .reset_index(drop=True)
)

X_normal_train, X_normal_calibration = train_test_split(
    normal_data,
    test_size=0.20,
    random_state=42
)

train_hashes = pd.util.hash_pandas_object(
    X_normal_train, index=False
)

calibration_hashes = pd.util.hash_pandas_object(
    X_normal_calibration, index=False
)

overlap = set(train_hashes).intersection(
    set(calibration_hashes)
)

print("Unique normal records:", len(normal_data))
print("Normal training shape:", X_normal_train.shape)
print("Normal calibration shape:", X_normal_calibration.shape)
print("Feature-pattern overlap:", len(overlap))

assert len(overlap) == 0

Unique normal records: 51890
Normal training shape: (41512, 41)
Normal calibration shape: (10378, 41)
Feature-pattern overlap: 0


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import IsolationForest

numerical_columns = (
    X_normal_train.select_dtypes(include="number")
    .columns.tolist()
)

categorical_columns = (
    X_normal_train.select_dtypes(exclude="number")
    .columns.tolist()
)

anomaly_preprocessor = ColumnTransformer([
    (
        "numerical",
        SimpleImputer(strategy="median"),
        numerical_columns
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ))
        ]),
        categorical_columns
    )
])

anomaly_pipeline = Pipeline([
    ("preprocessor", anomaly_preprocessor),
    ("detector", IsolationForest(
        n_estimators=200,
        max_samples=256,
        contamination="auto",
        random_state=42,
        n_jobs=2
    ))
])

In [3]:
import numpy as np
from time import perf_counter

start_time = perf_counter()

anomaly_pipeline.fit(X_normal_train)

print(
    f"Training time: {perf_counter() - start_time:.2f} seconds"
)

# Negate scores so that higher = more anomalous
calibration_scores = -anomaly_pipeline.score_samples(
    X_normal_calibration
)

print(
    "Calibration score percentiles:",
    np.quantile(
        calibration_scores,
        [0.00, 0.50, 0.95, 0.99, 1.00]
    )
)

Training time: 1.14 seconds
Calibration score percentiles: [0.33581588 0.39235417 0.52280413 0.57520933 0.63391709]


In [4]:
target_normal_flag_rate = 0.05

anomaly_threshold = float(
    np.quantile(
        calibration_scores,
        1 - target_normal_flag_rate
    )
)

calibration_flags = (
    calibration_scores > anomaly_threshold
)

print(f"Anomaly threshold: {anomaly_threshold:.6f}")
print(
    f"Calibration normal flag rate: "
    f"{calibration_flags.mean():.2%}"
)
print("Flagged normal records:", int(calibration_flags.sum()))

Anomaly threshold: 0.522804
Calibration normal flag rate: 5.00%
Flagged normal records: 519


In [5]:
from sklearn.metrics import confusion_matrix, roc_auc_score

test_df = pd.read_csv(
    project_root / "data/raw/UNSW_NB15_testing-set.csv"
)

X_anomaly_test = test_df[feature_columns]
y_anomaly_test = test_df["label"].to_numpy()

# Higher score = more anomalous
test_anomaly_scores = -anomaly_pipeline.score_samples(
    X_anomaly_test
)

test_anomaly_flags = (
    test_anomaly_scores > anomaly_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_anomaly_test,
    test_anomaly_flags,
    labels=[0, 1]
).ravel()

print("Normal records correctly unflagged:", tn)
print("Normal records flagged as suspicious:", fp)
print("Attack records flagged as suspicious:", tp)
print("Attack records not flagged:", fn)

print(f"\nNormal false-positive rate: {fp / (fp + tn):.2%}")
print(f"Attack detection recall: {tp / (tp + fn):.2%}")

print(
    "Anomaly ROC-AUC:",
    round(
        roc_auc_score(y_anomaly_test, test_anomaly_scores),
        4
    )
)

Normal records correctly unflagged: 33497
Normal records flagged as suspicious: 3503
Attack records flagged as suspicious: 16191
Attack records not flagged: 29141

Normal false-positive rate: 9.47%
Attack detection recall: 35.72%
Anomaly ROC-AUC: 0.7958


In [6]:
from sklearn.metrics import confusion_matrix

# Use the saved binary model on these same test rows
binary_pipeline = binary_artifact["pipeline"]

attack_column = list(
    binary_pipeline.named_steps["classifier"].classes_
).index(1)

binary_scores = binary_pipeline.predict_proba(
    test_df[binary_artifact["input_columns"]]
)[:, attack_column]

binary_flags = (
    binary_scores >= binary_artifact["threshold"]
)

anomaly_flags = test_anomaly_flags.astype(bool)
actual_attack = y_anomaly_test == 1

# Anomaly warnings added only where binary model said Normal
additional_flags = (~binary_flags) & anomaly_flags

recovered_attacks = (
    additional_flags & actual_attack
).sum()

additional_false_alarms = (
    additional_flags & (~actual_attack)
).sum()

# Evaluate the proposed OR combination
combined_flags = binary_flags | anomaly_flags

tn, fp, fn, tp = confusion_matrix(
    y_anomaly_test,
    combined_flags.astype(int),
    labels=[0, 1]
).ravel()

print("Binary-missed attacks recovered:", int(recovered_attacks))
print("Additional normal false alarms:", int(additional_false_alarms))

print("\nCombined false alarms:", fp)
print("Combined missed attacks:", fn)
print(f"Combined attack recall: {tp / (tp + fn):.2%}")
print(f"Combined false-positive rate: {fp / (fp + tn):.2%}")

Binary-missed attacks recovered: 114
Additional normal false alarms: 2506

Combined false alarms: 12736
Combined missed attacks: 434
Combined attack recall: 99.04%
Combined false-positive rate: 34.42%


In [7]:
import joblib
import numpy as np
import sklearn

anomaly_artifact = {
    "pipeline": anomaly_pipeline,
    "threshold": float(anomaly_threshold),
    "input_columns": feature_columns,
    "score_definition": "negative_score_samples",
    "flag_rule": "score > threshold",
    "role": "Supplementary warning; does not override binary prediction",
    "sklearn_version": sklearn.__version__,
    "model_name": "Isolation Forest normal-traffic baseline",
    "calibration_target_flag_rate": 0.05
}

model_directory = project_root / "models"
model_directory.mkdir(parents=True, exist_ok=True)

anomaly_path = model_directory / "anomaly_isolation_forest.joblib"

joblib.dump(anomaly_artifact, anomaly_path)

# Verify saved and reloaded scores
loaded_anomaly = joblib.load(anomaly_path)

sample = X_normal_calibration.iloc[:10][feature_columns]

original_scores = -anomaly_pipeline.score_samples(sample)
reloaded_scores = -loaded_anomaly["pipeline"].score_samples(sample)

np.testing.assert_allclose(
    original_scores,
    reloaded_scores
)

np.testing.assert_array_equal(
    original_scores > anomaly_threshold,
    reloaded_scores > loaded_anomaly["threshold"]
)

print("Saved model:", anomaly_path)
print("Anomaly reload check passed")

Saved model: D:\PROGRAMMING\NetGuard AI\netguard_ai\models\anomaly_isolation_forest.joblib
Anomaly reload check passed
